# Day 13 — cross-paper agent (LangGraph)

A second LangGraph graph for questions that need **several papers** (the frozen within-paper agent is not changed):

    retrieve_papers  ->  synthesize
    (hybrid search + reranker,       (LLM: one bullet per relevant paper,
     top 10 papers x 2 passages)      [E#] citations after every claim)

It is measured on the 19 frozen cross-paper questions:
* **paper coverage** (no LLM needed to score): how many required papers were shown to the LLM and how many it cited,
  and how many cited papers are correct (required or acceptable);
* **judge** (gpt-oss-20b): does every citation support its sentence, is the answer faithful.

**Quota:** the agent uses gpt-oss-120b, the same Groq quota as the 100-question test run. Finish the test run's
agent cell first; this run needs ~6-7k tokens per question (~130k for 19), so it may need `--resume` the next day.
Use a **T4 GPU** runtime (the reranker runs for every question).

In [8]:
# WHAT: Mounts Google Drive and makes the research-engine folder the working directory.
# WHY:  All commands below run from the project root. Run this first in every new session.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
PROJECT = glob.glob('/content/drive/MyDrive/research-engine')[0]
os.chdir(PROJECT)
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/research-engine


In [9]:
# WHAT: Installs the project's packages (run again after every runtime restart).
# WHY:  A fresh Colab session doesn't have them.
!pip install -q -r requirements.txt

In [10]:
# WHAT: Copies the Groq key from Colab Secrets into an environment variable and checks for a GPU.
# WHY:  The `!python` commands can only see environment variables; the reranker needs the GPU to be fast.
from google.colab import userdata
import torch
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()
print("key loaded:", len(os.environ["GROQ_API_KEY"]) > 0, "| GPU:", torch.cuda.is_available())

key loaded: True | GPU: True


In [ ]:
# WHAT: Runs only the retrieval half of the cross-paper agent on the HotpotQA question and prints the 10 papers
#       the LLM would see. It makes no LLM call.
# WHY:  Checks for free that the new code imports and that the GPU reranker and FAISS work, before any quota is
#       spent on real answers.
from agent.cross_paper import CrossPaperAgent
agent = CrossPaperAgent()
q = "Which papers run experiments on the multi-hop question answering benchmark HotpotQA, and what does each of them propose?"
papers = agent.retriever.retrieve(q, rerank=True)
for p in papers:
    print(f"{p['rank']:>2}. {p['paper_id']}  {p['title'][:80]}  ({len(p['chunks'])} passages)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

 1. 2004.02393  Learning to Recover Reasoning Chains for Multi-Hop Question Answering via Cooper  (2 passages)
 2. 1910.04601  RC-QED: Evaluating Natural Language Derivations in Multi-Hop Reading Comprehensi  (2 passages)
 3. 1909.08041  Revealing the Importance of Semantic Retrieval for Machine Reading at Scale  (2 passages)
 4. 2004.02143  Reinforced Multi-task Approach for Multi-hop Question Generation  (2 passages)
 5. 2002.09758  Unsupervised Question Decomposition for Question Answering  (2 passages)
 6. 1908.09137  Propagate-Selector: Detecting Supporting Sentences for Question Answering via Gr  (2 passages)
 7. 1905.10044  BoolQ: Exploring the Surprising Difficulty of Natural Yes/No Questions  (2 passages)
 8. 1911.09241  Assessing the Benchmarking Capacity of Machine Reading Comprehension Datasets  (2 passages)
 9. 1706.07179  RelNet: End-to-End Modeling of Entities & Relations  (2 passages)
10. 1808.09920  Question Answering by Reasoning Across Documents with Graph Convolutio

## Step 1 — one question, to see what the agent does

In [ ]:
# WHAT: Runs the cross-paper agent on one question and prints the papers it found, its answer and what it cited.
# WHY:  To read one real answer before running all 19. Costs one LLM call (~6k tokens).
from agent.cross_paper import CrossPaperAgent
agent = CrossPaperAgent()
out = agent.run("Which papers run experiments on the multi-hop question answering benchmark HotpotQA, and what does each of them propose?")
print("Papers shown to the LLM:")
for p in out["papers"]:
    print(f"  {p['rank']:>2}. {p['paper_id']}  {p['title'][:80]}")
print("\nAnswer:\n" + out["answer"])
print("\nCited papers:", out["cited_papers"], "| invalid labels:", out["invalid_labels"])
print("tokens:", out["tokens"], "|", out["secs"], "s")

Papers shown to the LLM:
   1. 2004.02393  Learning to Recover Reasoning Chains for Multi-Hop Question Answering via Cooper
   2. 1910.04601  RC-QED: Evaluating Natural Language Derivations in Multi-Hop Reading Comprehensi
   3. 1909.08041  Revealing the Importance of Semantic Retrieval for Machine Reading at Scale
   4. 2004.02143  Reinforced Multi-task Approach for Multi-hop Question Generation
   5. 2002.09758  Unsupervised Question Decomposition for Question Answering
   6. 1908.09137  Propagate-Selector: Detecting Supporting Sentences for Question Answering via Gr
   7. 1905.10044  BoolQ: Exploring the Surprising Difficulty of Natural Yes/No Questions
   8. 1911.09241  Assessing the Benchmarking Capacity of Machine Reading Comprehension Datasets
   9. 1706.07179  RelNet: End-to-End Modeling of Entities & Relations
  10. 1808.09920  Question Answering by Reasoning Across Documents with Graph Convolutional Networ

Answer:
- Learning to Recover Reasoning Chains for Multi‑Hop Question

## Step 2 — all 19 questions
Saves one line per question to `results/cross_paper_agent.jsonl`. If Groq's daily limit is reached it prints
**STOPPED** and keeps everything; run the same cell again the next day (`--resume`).

In [ ]:
# WHAT: Runs the cross-paper agent on the 19 frozen questions and scores paper coverage.
# WHY:  shown_rec = required papers the LLM saw; cited_rec = required papers the answer cites;
#       cited_prec = share of cited papers that are correct. ~20-30 s per question on a T4.
!python -m eval.run_cross_paper_agent_eval --resume

Loading weights: 100% 199/199 [00:00<00:00, 639.58it/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.npy
19 cross-paper questions | model=openai/gpt-oss-120b | rerank=True | 10 papers x 2 passages | resuming: 19 done

n=19
| metric | all | dataset | topic |
|---|---|---|---|
| retrieved_paper_recall | 0.8397 | 0.8857 | 0.7885 |
| cited_paper_recall | 0.6339 | 0.6738 | 0.5895 |
| cited_paper_precision | 0.9837 | 0.9857 | 0.9815 |
| avg_cited_papers | 5.05 | 4.7 | 5.44 |
| declined_rate | 0.0 | 0.0 | 0.0 |
| invalid_citation_rate | 0.0 | 0.0 | 0.0 |
| avg_prompt_tokens | 4269 | 4186 | 4360 |
| avg_completion_tokens | 442 | 410 | 477 |
| avg_secs | 30.17 | 27.17 | 33.51 |

Saved /content/drive/MyDrive/research-engine/results/cross_paper_agent.json and /content/drive/MyDrive/research-engine/results/cross_paper_agent.jsonl


## Step 3 — the judge (gpt-oss-20b, its own quota)

In [ ]:
# WHAT: Judges each saved answer: does every [E#] support its sentence, is the answer faithful, is what it says about
#       each paper supported by that paper's own passages?
# WHY:  Paper coverage says WHICH papers were cited; the judge checks that what was said about them is true.
!python -m eval.judge_cross_paper results/cross_paper_agent.jsonl --resume

19 answers | judge=openai/gpt-oss-20b
[XP01] cite_support=1.00 faithful=yes per_paper=1.00
[XP02] cite_support=1.00 faithful=yes per_paper=1.00
[XP03] cite_support=1.00 faithful=yes per_paper=1.00
[XP04] cite_support=1.00 faithful=yes per_paper=1.00
[XP05] cite_support=0.67 faithful=partly per_paper=1.00
[XP06] cite_support=1.00 faithful=yes per_paper=1.00
[XP07] cite_support=1.00 faithful=yes per_paper=1.00
[XP08] cite_support=1.00 faithful=yes per_paper=1.00
[XP09] cite_support=1.00 faithful=partly per_paper=1.00
[XP10] cite_support=1.00 faithful=yes per_paper=1.00
[XP11] cite_support=1.00 faithful=partly per_paper=1.00
[XP12] cite_support=1.00 faithful=yes per_paper=1.00
[XP13] cite_support=0.80 faithful=partly per_paper=1.00
[XP14] cite_support=1.00 faithful=partly per_paper=1.00
[XP15] cite_support=1.00 faithful=no per_paper=0.20
[XP16] cite_support=1.00 faithful=yes per_paper=1.00
[XP17] cite_support=1.00 faithful=yes per_paper=1.00
[XP18] cite_support=0.50 faithful=partly per_pa

In [ ]:
# WHAT: Prints the agent and judge summaries (all questions, dataset questions, topic questions).
# WHY:  The numbers for the results log and the CV.
import json
for f in ["results/cross_paper_agent.json", "results/cross_paper_agent_judge-gpt-oss-20b.json"]:
    if os.path.exists(f):
        d = json.load(open(f))
        print(f"\n== {f} (n={d['n']}, complete={d['complete']})")
        for k in d["summary"]["all"]:
            print(f"  {k:<30} " + "  ".join(f"{s}={d['summary'][s].get(k)}" for s in d["summary"]))


== results/cross_paper_agent.json (n=19, complete=True)
  retrieved_paper_recall         all=0.8397  dataset=0.8857  topic=0.7885
  cited_paper_recall             all=0.6339  dataset=0.6738  topic=0.5895
  cited_paper_precision          all=0.9837  dataset=0.9857  topic=0.9815
  avg_cited_papers               all=5.05  dataset=4.7  topic=5.44
  declined_rate                  all=0.0  dataset=0.0  topic=0.0
  invalid_citation_rate          all=0.0  dataset=0.0  topic=0.0
  avg_prompt_tokens              all=4269  dataset=4186  topic=4360
  avg_completion_tokens          all=442  dataset=410  topic=477
  avg_secs                       all=30.17  dataset=27.17  topic=33.51

== results/cross_paper_agent_judge-gpt-oss-20b.json (n=19, complete=True)
  judged_citation_precision      all=0.9456  dataset=0.9667  topic=0.9222
  faithfulness                   all=0.7895  dataset=0.9  topic=0.6667
  per_paper_support              all=0.9316  dataset=1.0  topic=0.8556
  declined_rate              

### Which required papers did the LLM skip? (no LLM)

In [ ]:
# WHAT: For every required paper that the LLM was SHOWN but did NOT cite, prints the paper and the start of the
#       2 passages it saw. No LLM call.
# WHY:  The answer cites ~3 of every 4 required papers it sees (0.63 vs 0.84). If these passages clearly say
#       "we use <dataset>" / are about the topic, skipping them is an agent error; if they don't, the skip is defensible.
#       Result (v1): 31 skipped; ~23 passages plainly show the dataset/topic, and 8/19 answers stop at exactly 5 papers.
import json
rows = [json.loads(l) for l in open("results/cross_paper_agent.jsonl")]
n_missed = 0
for r in rows:
    shown = {p["paper_id"]: p["title"] for p in r["papers"]}
    missed = [p for p in r["required_papers"] if p in shown and p not in r["cited_papers"]]
    if not missed:
        continue
    print(f"\n===== {r['id']}: {r['question'][:90]}")
    for pid in missed:
        n_missed += 1
        print(f"  -- {pid}  {shown[pid][:70]}")
        for e in r["evidence"]:
            if e["paper_id"] == pid:
                print(f"     [{e['label']}] ({e['section'][:30]}) {e['text'][:220].replace(chr(10), ' ')}")
print(f"\n{n_missed} required papers shown but not cited")

### Why are 7 answers not fully faithful? (no LLM)

In [ ]:
# WHAT: Prints the answer and the judge's full verdict for the 7 answers not judged fully faithful.
# WHY:  To see WHY faithfulness is 0.79: uncited comparison sentences, real overclaims, or judge mistakes
#       (XP15 has every citation "yes" but faithful "no", which contradicts itself). No LLM call.
import json
judged = {j["id"]: j for j in map(json.loads, open("results/cross_paper_agent_judge-gpt-oss-20b.jsonl"))}
for qid in ["XP05", "XP09", "XP11", "XP13", "XP14", "XP15", "XP18"]:
    j = judged[qid]
    print(f"\n===== {qid} faithful={j['judge']['faithful']} | {j['question'][:80]}")
    print(j["answer"])
    print("  citations:  ", j["judge"]["citations"])
    print("  per_paper:  ", j["judge"]["per_paper"])
    print("  unsupported:", j["judge"]["unsupported"])
    print("  note:       ", j["judge"]["note"])


===== XP05 faithful=partly | Which papers evaluate answer selection or question answering models on WikiQA, a
- Analysis of Wikipedia-based Corpora for Question Answering: evaluates answer‑selection models on WikiQA and reports mean average precision (MAP) and mean reciprocal rank (MRR) as the evaluation metrics[E1].  
- ISS‑MULT: Intelligent Sample Selection for Multi‑Task Learning in Question Answering: evaluates answer‑selection on WikiQA (among other datasets) and uses MAP and MRR to report results[E3][E4].
  citations:   {'E1': 'yes', 'E3': 'yes', 'E4': 'no'}
  per_paper:   {'1801.02073': 'yes', '1708.02267': 'yes'}
  unsupported: ['The claim that ISS‑MULT evaluates answer‑selection on WikiQA (among other datasets) is not directly supported by the cited passages.']
  note:        

===== XP09 faithful=partly | Which papers evaluate language models on WikiText-2 or WikiText-103, and what do
- Character n‑gram Embeddings to Improve RNN Language Models: Shows that the proposed chara

## Step 4 — judge fix, then re-judge v1
The inspection above showed a judge **scoring bug**: in XP15 the judge gave a verdict for only 1 of 8 cited labels and the
script silently dropped the other 7 (citation support showed 1.00 from one label); XP13 had one label without a verdict.
The fix (`research-engine-day13-v2.zip`): a reply that leaves a cited label without yes/partly/no is retried once, then
saved as a judge error; `--resume` re-judges those rows. The zip also adds the optional v2 prompt (v1 stays the default).

In [11]:
# WHAT: Counts answers in the 100-question TEST judge file where a cited label got no verdict, and recomputes judged
#       citation precision counting those labels as "no" (worst case).
# WHY:  The within-paper judge uses the same scoring idea as the cross-paper judge, so this checks whether the headline
#       test result (0.956) is affected. Read-only: nothing is re-run or changed.
import json
rows = [json.loads(l) for l in open("results/agent_eval_test_n100_v2_judge-gpt-oss-20b.jsonl")]
SC = {"yes": 1.0, "partly": 0.5, "no": 0.0}
bad, as_reported, worst = [], [], []
for r in rows:
    c = (r.get("judge") or {}).get("citations") or {}
    if isinstance(c, list):
        c = {str(x.get("label")): x.get("supports") for x in c if isinstance(x, dict)}
    if not c:
        continue
    vals = list(c.values())
    if any(v not in SC for v in vals):
        bad.append(r["id"])
    known = [SC[v] for v in vals if v in SC]
    if known:
        as_reported.append(sum(known) / len(known))
    worst.append(sum(SC.get(v, 0.0) for v in vals) / len(vals))
print("answers with a cited label left without a verdict:", len(bad), bad)
print("judged citation precision, as reported (missing labels dropped):", round(sum(as_reported) / len(as_reported), 3))
print("judged citation precision, worst case (missing = no):          ", round(sum(worst) / len(worst), 3))

answers with a cited label left without a verdict: 0 []
judged citation precision, as reported (missing labels dropped): 0.956
judged citation precision, worst case (missing = no):           0.956


In [13]:
# WHAT: Re-runs the judge with --resume. Complete rows are kept; rows saved with a missing citation verdict
#       (XP15, XP13) are judged again with the fixed judge.
# WHY:  Gives the corrected v1 judge numbers. The agent's answers are NOT regenerated, so no 120b quota is used.
!python -m eval.judge_cross_paper results/cross_paper_agent.jsonl --resume

re-judging 2 rows with an error or a missing citation verdict: XP13, XP15
19 answers | judge=openai/gpt-oss-20b | resuming: 17 judged
[XP13] cite_support=1.00 faithful=yes per_paper=1.00
[XP15] cite_support=0.12 faithful=no per_paper=0.20

judge=openai/gpt-oss-20b n=19
| metric | all | dataset | topic |
|---|---|---|---|
| judged_citation_precision | 0.9101 | 0.9667 | 0.8472 |
| faithfulness | 0.8158 | 0.9 | 0.7222 |
| per_paper_support | 0.9316 | 1.0 | 0.8556 |
| declined_rate | 0.0 | 0.0 | 0.0 |
| judge_errors | 0 | 0 | 0 |
| avg_judge_prompt_tokens | 2253 | 2127 | 2393 |
| avg_judge_completion_tokens | 189 | 185 | 194 |

Not fully faithful:
- XP05 faithful=partly | 
- XP09 faithful=partly | The summary of which papers evaluate WikiText‑2 or WikiText‑103 is partially unsupported.
- XP11 faithful=partly | 
- XP14 faithful=partly | 
- XP15 faithful=no | Only E2 is supported by its passage
- XP18 faithful=partly | E2 passage does not mention RNN or embeddings

Saved results/cross_paper_

## Step 5 — v2 prompt (written after the v1 error analysis on these same 19 questions)
v2 changes only the synthesis prompt: check every paper one by one and include **every** paper whose own passages show
it addresses the question (no limit on bullets; papers are not ordered by relevance); describe a paper only from its own
passages, not its title; every comparison sentence must be cited or left out.
Because v2 was written after looking at these questions, its numbers are **not held out**: v1 stays the primary result
and v2 is reported next to it. v2 is kept only if cited-paper precision stays high (v1 0.984). Coverage cannot exceed
the 0.840 of required papers that retrieval shows the LLM.

In [14]:
# WHAT: Runs the cross-paper agent with the v2 prompt on the 19 questions; saves results/cross_paper_agent_v2.json(l).
# WHY:  Measures whether "include every relevant paper" raises cited_paper_recall (v1 0.634) without losing precision.
#       ~90k gpt-oss-120b tokens; if it prints STOPPED (daily limit), run this cell again the next day.
!python -m eval.run_cross_paper_agent_eval --prompt v2 --tag v2 --resume

Loading weights: 100% 199/199 [00:00<00:00, 5814.99it/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.npy
19 cross-paper questions | model=openai/gpt-oss-120b | prompt=v2 | rerank=True | 10 papers x 2 passages
FAISS IndexFlatIP built in 0.04 s (28541 vectors, dim 384)
Loading weights: 100% 393/393 [00:00<00:00, 4879.38it/s]
[XP01] shown_rec=1.00 cited_rec=0.83 cited_prec=1.00 papers_cited=5 tokens=5290 12.88s  Which papers run experiments on the multi-hop ques
[XP02] shown_rec=0.75 cited_rec=0.62 cited_prec=1.00 papers_cited=5 tokens=4875 24.04s  Which papers use the MultiWOZ dataset in their exp
[XP03] shown_rec=0.88 cited_rec=0.88 cited_prec=0.89 papers_cited=9 tokens=4556 39.8s  Which papers train or evaluate dialogue models on 
[XP04] shown_rec=0.86 cited_rec=0.57 cited_prec=1.00 papers_cited=4 tokens=5329 19.65s  Which papers report experiments on the MS MARCO da
[XP05] shown_rec=0.75 cited_rec=0.25 cited_prec=1.00 papers

In [15]:
# WHAT: Judges the v2 answers with the fixed judge (gpt-oss-20b).
# WHY:  Checks that the extra papers v2 cites are described faithfully, and whether cited comparisons raise faithfulness.
!python -m eval.judge_cross_paper results/cross_paper_agent_v2.jsonl --resume

19 answers | judge=openai/gpt-oss-20b
[XP01] cite_support=0.80 faithful=partly per_paper=0.80
[XP02] cite_support=1.00 faithful=yes per_paper=1.00
[XP03] cite_support=1.00 faithful=no per_paper=1.00
[XP04] cite_support=1.00 faithful=yes per_paper=1.00
[XP05] cite_support=1.00 faithful=yes per_paper=1.00
[XP06] cite_support=1.00 faithful=yes per_paper=1.00
[XP07] cite_support=1.00 faithful=yes per_paper=1.00
[XP08] cite_support=1.00 faithful=yes per_paper=1.00
[XP09] cite_support=1.00 faithful=partly per_paper=1.00
[XP10] cite_support=1.00 faithful=yes per_paper=1.00
[XP11] cite_support=1.00 faithful=partly per_paper=1.00
[XP12] cite_support=1.00 faithful=yes per_paper=1.00
[XP13] cite_support=1.00 faithful=partly per_paper=1.00
[XP14] cite_support=1.00 faithful=yes per_paper=1.00
[XP15] cite_support=1.00 faithful=yes per_paper=1.00
[XP16] cite_support=1.00 faithful=yes per_paper=1.00
[XP17] cite_support=1.00 faithful=yes per_paper=1.00
[XP18] cite_support=1.00 faithful=yes per_paper=1.

In [16]:
# WHAT: Puts v1 and v2 side by side: paper coverage, judge scores, and per question cited recall / papers cited.
# WHY:  The numbers for the results cell, the report and the CV. No LLM call.
import json
def summ(f):
    return json.load(open(f))["summary"]["all"] if os.path.exists(f) else {}
for a, b in [("results/cross_paper_agent.json", "results/cross_paper_agent_v2.json"),
             ("results/cross_paper_agent_judge-gpt-oss-20b.json", "results/cross_paper_agent_v2_judge-gpt-oss-20b.json")]:
    A, B = summ(a), summ(b)
    print(f"\n{a.split('/')[-1]}  vs  v2")
    print(f"  {'metric':<30} {'v1':>8} {'v2':>8}")
    for k in A:
        print(f"  {k:<30} {str(A.get(k)):>8} {str(B.get(k)):>8}")
v1 = {r["id"]: r for r in map(json.loads, open("results/cross_paper_agent.jsonl"))}
v2 = {r["id"]: r for r in map(json.loads, open("results/cross_paper_agent_v2.jsonl"))} if os.path.exists("results/cross_paper_agent_v2.jsonl") else {}
print(f"\n  {'id':<5} {'shown':>6} {'cited v1':>9} {'cited v2':>9} {'#papers v1':>11} {'#papers v2':>11}")
for qid, r in v1.items():
    m1, m2 = r["metrics"], v2.get(qid, {}).get("metrics", {})
    f = lambda v: "-" if v is None else f"{v:.2f}"
    print(f"  {qid:<5} {f(m1['retrieved_paper_recall']):>6} {f(m1['cited_paper_recall']):>9} {f(m2.get('cited_paper_recall')):>9} "
          f"{m1['n_cited_papers']:>11} {str(m2.get('n_cited_papers', '-')):>11}")


cross_paper_agent.json  vs  v2
  metric                               v1       v2
  retrieved_paper_recall           0.8397   0.8397
  cited_paper_recall               0.6339   0.6565
  cited_paper_precision            0.9837   0.9788
  avg_cited_papers                   5.05     5.53
  declined_rate                       0.0      0.0
  invalid_citation_rate               0.0      0.0
  avg_prompt_tokens                  4269     4371
  avg_completion_tokens               442      538
  avg_secs                          30.17    32.19

cross_paper_agent_judge-gpt-oss-20b.json  vs  v2
  metric                               v1       v2
  judged_citation_precision        0.9101   0.9895
  faithfulness                     0.8158   0.8158
  per_paper_support                0.9316   0.9895
  declined_rate                       0.0      0.0
  judge_errors                          0        0
  avg_judge_prompt_tokens            2253     2436
  avg_judge_completion_tokens         189      217


## Results
**v1** (one run, 2026-10-17; 19 frozen questions; primary, untuned result)
* Paper coverage: required papers shown to the LLM 0.840, cited in the answer 0.634; cited-paper precision 0.984; 0 declines, 0 invalid citations; 4,269 + 442 tokens/question.
* Judge (gpt-oss-20b, re-judged after the missing-verdict fix): citation support 0.910 (dataset 0.967, topic 0.847), faithfulness 0.816, per-paper support 0.932.
* Diagnostic: 31 required papers shown but not cited; ~23 had passages that plainly show the dataset/topic; 8/19 answers stop at exactly 5 papers.
* Judge bug found and fixed: cited labels left without a verdict were silently dropped (XP15 scored 1.00 from 1 of 8 labels; re-judged: 0.12). The 100-question test judge is not affected (0 such answers).

**v2** (prompt revised after the v1 error analysis on these same questions, so NOT held out; used in the app)
* Paper coverage: cited 0.657 (v1 0.634), cited-paper precision 0.979, 5.53 papers per answer; 4,371 + 538 tokens/question.
* Judge: citation support 0.990, per-paper support 0.990, faithfulness 0.816 (unchanged).
* v2 fixed claims taken from paper titles (XP15 0.12 -> 1.00, XP18 0.50 -> 1.00) but not under-inclusion (~1 in 4 required papers shown are still skipped). No further tuning on these questions; a per-paper map-reduce node is future work.